# 从零复现 DETR：集合预测、手写 QKV 与二分匹配

本 Notebook 不调用检测器、`torchvision.models`、`nn.MultiheadAttention` 或 `nn.Transformer`。我们从 Tiny CNN backbone 开始，显式实现二维位置编码、multi-head Q/K/V attention、encoder、decoder、object queries、类别头与归一化 box head；再用穷举法实现小规模最优二分匹配和 set loss。

除 forward 之外，还要验证：预测顺序不影响集合损失、空目标图像只训练 no-object、padding 像素不会泄漏、box 坐标合法、梯度贯通 backbone/queries/heads、推理过滤 no-object，以及模型制品绑定标签和预处理。

数据完全离线合成、固定 seed、CPU 单线程。微型穷举匹配仅用于教学 oracle，生产规模必须换成经过验证的 Hungarian 实现。

## 1. DETR 的计算图与集合语义

```text
image + pixel padding mask
  -> TinyCNN [B,D,H',W']
  -> flatten + 2D position [B,S,D]
  -> manual encoder self-attention
  -> learned object queries + manual decoder self/cross-attention
  -> class logits [B,Q,K+1] + normalized cxcywh boxes [B,Q,4]
  -> one-to-one matching -> set loss
```

$Q$ 个 query 没有固定“第一个物体、第二个物体”语义。匹配在所有 query 与真实目标间选择一对一分配，未匹配 query 的类别为 $\varnothing$（no-object）。因此训练目标是集合，而不是按数组下标对齐。

In [ ]:
import warnings  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

from copy import deepcopy  # 导入本单元所需的依赖。
from hashlib import sha256  # 导入本单元所需的依赖。
from itertools import permutations  # 导入本单元所需的依赖。
from types import MappingProxyType  # 导入本单元所需的依赖。
import io  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED = 350728  # 计算并保存当前步骤的中间状态。
random.seed(SEED)  # 执行当前语句以推进本节示例。
np.random.seed(SEED)  # 执行当前语句以推进本节示例。
torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
torch.use_deterministic_algorithms(True)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE = torch.device("cpu")  # 计算并保存当前步骤的中间状态。

assert DEVICE.type == "cpu"  # 用受控断言验证关键不变量。
assert torch.get_num_threads() == 1  # 用受控断言验证关键不变量。
print({"torch": torch.__version__, "device": str(DEVICE), "seed": SEED})  # 执行当前语句以推进本节示例。

## 2. Tiny CNN backbone 与 padding mask

卷积 backbone 把 `[B,1,16,16]` 变成 `[B,D,4,4]`。输入 padding mask 使用 `True=padding`；进入卷积前必须把 padding 像素清零，否则调用方改变无效区域也会改变边界特征。mask 随分辨率下采样时采用 max pooling：一个 feature cell 的感受野只要覆盖 padding，就保守地标为 padding。

真实 DETR 常用预训练 ResNet；这里手写小 backbone，只复现接口和梯度路径。

In [ ]:
class TinyCNNBackbone(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_channels=1, hidden_dim=16):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.in_channels = int(in_channels)  # 计算并保存当前步骤的中间状态。
        self.network = nn.Sequential(  # 计算并保存当前步骤的中间状态。
            nn.Conv2d(in_channels, 16, 3, stride=2, padding=1, bias=False),  # 计算并保存当前步骤的中间状态。
            nn.BatchNorm2d(16), nn.ReLU(inplace=False),  # 计算并保存当前步骤的中间状态。
            nn.Conv2d(16, hidden_dim, 3, stride=2, padding=1, bias=False),  # 计算并保存当前步骤的中间状态。
            nn.BatchNorm2d(hidden_dim), nn.ReLU(inplace=False),  # 计算并保存当前步骤的中间状态。
            nn.Conv2d(hidden_dim, hidden_dim, 3, stride=2, padding=1), nn.ReLU(inplace=False),  # 计算并保存当前步骤的中间状态。
        )  # 执行当前语句以推进本节示例。

    def propagate_padding_mask(self, pixel_padding_mask):  # 定义本节可复用的核心函数。
        if pixel_padding_mask.ndim != 3 or pixel_padding_mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
            raise ValueError("pixel padding mask must be bool [B,H,W]")  # 遇到非法合同立即显式失败。
        feature_mask = pixel_padding_mask.float().unsqueeze(1)  # 计算并保存当前步骤的中间状态。
        for layer in self.network:  # 遍历输入元素以累积或检查结果。
            if isinstance(layer, nn.Conv2d):  # 按当前条件选择后续控制路径。
                feature_mask = F.max_pool2d(feature_mask, kernel_size=layer.kernel_size,  # 计算并保存当前步骤的中间状态。
                                            stride=layer.stride, padding=layer.padding,  # 计算并保存当前步骤的中间状态。
                                            dilation=layer.dilation)  # 计算并保存当前步骤的中间状态。
        return feature_mask.squeeze(1).bool()  # 返回当前分支计算出的结果。

    def forward(self, images):  # 定义本节可复用的核心函数。
        if images.ndim != 4 or images.shape[1] != self.in_channels:  # 按当前条件选择后续控制路径。
            raise ValueError("backbone expects configured NCHW images")  # 遇到非法合同立即显式失败。
        return self.network(images)  # 返回当前分支计算出的结果。

backbone_probe = TinyCNNBackbone()  # 计算并保存当前步骤的中间状态。
backbone_input = torch.randn(2, 1, 16, 16, requires_grad=True)  # 计算并保存当前步骤的中间状态。
backbone_output = backbone_probe(backbone_input)  # 计算并保存当前步骤的中间状态。
backbone_output.mean().backward()  # 执行当前语句以推进本节示例。
assert backbone_output.shape == (2, 16, 2, 2)  # 用受控断言验证关键不变量。
assert backbone_input.grad is not None and float(backbone_input.grad.norm()) > 0  # 用受控断言验证关键不变量。
assert torch.isfinite(backbone_input.grad).all()  # 用受控断言验证关键不变量。

## 3. 二维正弦位置编码

纯 attention 对 token 排列是等变的，展平 feature map 后必须显式注入行列位置。令频率 $\omega_i=10000^{-2i/d}$，分别计算行、列的 sin/cos，再拼成 $D$ 维。`hidden_dim` 必须被 4 整除，因为 x/y 各占一半，其中又各分 sin/cos。

padding 位置的 encoding 置零；attention 中还会再次以 key padding mask 屏蔽其 logits。两道约束解决的问题不同：前者防止位置值进入残差，后者保证 softmax 不把概率分给无效 key。

In [ ]:
class Sine2DPositionEncoding(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, hidden_dim=16, temperature=10000.0):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if hidden_dim % 4 != 0:  # 按当前条件选择后续控制路径。
            raise ValueError("hidden_dim must be divisible by four")  # 遇到非法合同立即显式失败。
        self.hidden_dim = int(hidden_dim)  # 计算并保存当前步骤的中间状态。
        self.temperature = float(temperature)  # 计算并保存当前步骤的中间状态。

    def forward(self, padding_mask):  # 定义本节可复用的核心函数。
        if padding_mask.ndim != 3 or padding_mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
            raise ValueError("padding mask must be bool [B,H,W]")  # 遇到非法合同立即显式失败。
        batch, height, width = padding_mask.shape  # 计算并保存当前步骤的中间状态。
        quarter = self.hidden_dim // 4  # 计算并保存当前步骤的中间状态。
        frequency = self.temperature ** (-torch.arange(quarter, device=padding_mask.device) / quarter)  # 计算并保存当前步骤的中间状态。
        y = torch.arange(height, device=padding_mask.device, dtype=torch.float32)[:, None] * frequency[None, :]  # 计算并保存当前步骤的中间状态。
        x = torch.arange(width, device=padding_mask.device, dtype=torch.float32)[:, None] * frequency[None, :]  # 计算并保存当前步骤的中间状态。
        y_encoding = torch.cat([y.sin(), y.cos()], dim=-1)[:, None, :].expand(height, width, -1)  # 计算并保存当前步骤的中间状态。
        x_encoding = torch.cat([x.sin(), x.cos()], dim=-1)[None, :, :].expand(height, width, -1)  # 计算并保存当前步骤的中间状态。
        position = torch.cat([y_encoding, x_encoding], dim=-1)  # 计算并保存当前步骤的中间状态。
        position = position.unsqueeze(0).expand(batch, -1, -1, -1).clone()  # 计算并保存当前步骤的中间状态。
        return position.masked_fill(padding_mask[..., None], 0.0)  # 返回当前分支计算出的结果。

position_encoder = Sine2DPositionEncoding(16)  # 计算并保存当前步骤的中间状态。
position_mask = torch.zeros(2, 4, 5, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
position_mask[:, -1, -1] = True  # 计算并保存当前步骤的中间状态。
position = position_encoder(position_mask)  # 计算并保存当前步骤的中间状态。
assert position.shape == (2, 4, 5, 16)  # 用受控断言验证关键不变量。
assert torch.equal(position[:, -1, -1], torch.zeros(2, 16))  # 用受控断言验证关键不变量。
assert not torch.equal(position[:, 0, 0], position[:, 0, 1])  # 用受控断言验证关键不变量。

## 4. 手写 multi-head QKV attention

对 query $Q\in\mathbb{R}^{T_q\times D}$ 和 memory $K,V\in\mathbb{R}^{T_k\times D}$：

$$A=\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_h}}+M\right),\qquad Z=AV.$$

reshape 后 attention 为 `[B,heads,Tq,Tk]`。mask 在 softmax 前把 padding key 设为负无穷；若某个样本所有 key 都被 mask，应 fail closed，而不是产生整行 NaN。下面的恒等投影 oracle 与显式矩阵乘法对齐缩放因子和 softmax 轴。

In [ ]:
class ManualMultiHeadAttention(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, hidden_dim, num_heads):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if hidden_dim % num_heads != 0:  # 按当前条件选择后续控制路径。
            raise ValueError("hidden_dim must divide num_heads")  # 遇到非法合同立即显式失败。
        self.hidden_dim = int(hidden_dim)  # 计算并保存当前步骤的中间状态。
        self.num_heads = int(num_heads)  # 计算并保存当前步骤的中间状态。
        self.head_dim = hidden_dim // num_heads  # 计算并保存当前步骤的中间状态。
        self.scale = self.head_dim ** -0.5  # 计算并保存当前步骤的中间状态。
        self.query = nn.Linear(hidden_dim, hidden_dim)  # 计算并保存当前步骤的中间状态。
        self.key = nn.Linear(hidden_dim, hidden_dim)  # 计算并保存当前步骤的中间状态。
        self.value = nn.Linear(hidden_dim, hidden_dim)  # 计算并保存当前步骤的中间状态。
        self.output = nn.Linear(hidden_dim, hidden_dim)  # 计算并保存当前步骤的中间状态。

    def _split(self, tensor):  # 定义本节可复用的核心函数。
        batch, tokens, _ = tensor.shape  # 计算并保存当前步骤的中间状态。
        return tensor.reshape(batch, tokens, self.num_heads, self.head_dim).transpose(1, 2)  # 返回当前分支计算出的结果。

    def forward(self, query, key, value, key_padding_mask=None, return_weights=False):  # 定义本节可复用的核心函数。
        if not (query.ndim == key.ndim == value.ndim == 3):  # 按当前条件选择后续控制路径。
            raise ValueError("attention expects [B,T,D]")  # 遇到非法合同立即显式失败。
        if query.shape[0] != key.shape[0] or key.shape != value.shape:  # 按当前条件选择后续控制路径。
            raise ValueError("attention batch/key/value mismatch")  # 遇到非法合同立即显式失败。
        if query.shape[-1] != self.hidden_dim or key.shape[-1] != self.hidden_dim:  # 按当前条件选择后续控制路径。
            raise ValueError("attention hidden dimension mismatch")  # 遇到非法合同立即显式失败。
        q, k, v = self._split(self.query(query)), self._split(self.key(key)), self._split(self.value(value))  # 计算并保存当前步骤的中间状态。
        scores = (q @ k.transpose(-2, -1)) * self.scale  # 计算并保存当前步骤的中间状态。
        if key_padding_mask is not None:  # 按当前条件选择后续控制路径。
            if key_padding_mask.shape != key.shape[:2] or key_padding_mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
                raise ValueError("invalid key padding mask")  # 遇到非法合同立即显式失败。
            if key_padding_mask.all(dim=1).any():  # 按当前条件选择后续控制路径。
                raise ValueError("an example cannot mask every key")  # 遇到非法合同立即显式失败。
            scores = scores.masked_fill(key_padding_mask[:, None, None, :], float("-inf"))  # 计算并保存当前步骤的中间状态。
        weights = scores.softmax(dim=-1)  # 计算并保存当前步骤的中间状态。
        context = (weights @ v).transpose(1, 2).contiguous().reshape(query.shape[0], query.shape[1], self.hidden_dim)  # 计算并保存当前步骤的中间状态。
        result = self.output(context)  # 计算并保存当前步骤的中间状态。
        return (result, weights) if return_weights else result  # 返回当前分支计算出的结果。

attention_oracle = ManualMultiHeadAttention(4, 1).eval()  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    for linear in (attention_oracle.query, attention_oracle.key,  # 遍历输入元素以累积或检查结果。
                   attention_oracle.value, attention_oracle.output):  # 执行当前语句以推进本节示例。
        linear.weight.copy_(torch.eye(4)); linear.bias.zero_()  # 执行当前语句以推进本节示例。
oracle_tokens = torch.tensor([[[1., 0., 0., 0.], [0., 2., 0., 0.]]])  # 计算并保存当前步骤的中间状态。
oracle_result, oracle_weights = attention_oracle(oracle_tokens, oracle_tokens, oracle_tokens,  # 计算并保存当前步骤的中间状态。
                                                  return_weights=True)  # 返回当前分支计算出的结果。
expected_weights = ((oracle_tokens @ oracle_tokens.transpose(1, 2)) / math.sqrt(4)).softmax(-1)  # 计算并保存当前步骤的中间状态。
expected_result = expected_weights @ oracle_tokens  # 计算并保存当前步骤的中间状态。
assert torch.allclose(oracle_weights[:, 0], expected_weights)  # 用受控断言验证关键不变量。
assert torch.allclose(oracle_result, expected_result)  # 用受控断言验证关键不变量。
assert torch.allclose(oracle_weights.sum(-1), torch.ones(1, 1, 2))  # 用受控断言验证关键不变量。

## 5. 显式 Encoder、Decoder 与 object queries

Encoder 让图像 token 相互聚合；Decoder 先让 object queries 自注意，再以 query 为 Q、encoder memory 为 K/V 做 cross-attention。learned query embedding 是“检测槽位”的身份信息，初始 decoder content 为零。

类别头输出 $K+1$ 类，最后一类固定为 no-object。box MLP 末尾 `sigmoid`，得到相对图像大小的 `(cx,cy,w,h)\in(0,1)^4`。这只保证数值范围，不保证 box 边界完全落在图像内；后处理转换到 xyxy 后仍应 clip。

In [ ]:
class FeedForward(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, hidden_dim):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.network = nn.Sequential(nn.Linear(hidden_dim, 2 * hidden_dim), nn.ReLU(),  # 计算并保存当前步骤的中间状态。
                                     nn.Linear(2 * hidden_dim, hidden_dim))  # 执行当前语句以推进本节示例。
    def forward(self, x):  # 定义本节可复用的核心函数。
        return self.network(x)  # 返回当前分支计算出的结果。

class EncoderLayer(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, hidden_dim=16, num_heads=4):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.attention = ManualMultiHeadAttention(hidden_dim, num_heads)  # 计算并保存当前步骤的中间状态。
        self.norm1, self.norm2 = nn.LayerNorm(hidden_dim), nn.LayerNorm(hidden_dim)  # 计算并保存当前步骤的中间状态。
        self.ffn = FeedForward(hidden_dim)  # 计算并保存当前步骤的中间状态。

    def forward(self, x, position, padding_mask):  # 定义本节可复用的核心函数。
        qk = x + position  # 计算并保存当前步骤的中间状态。
        x = self.norm1(x + self.attention(qk, qk, x, padding_mask))  # 计算并保存当前步骤的中间状态。
        return self.norm2(x + self.ffn(x))  # 返回当前分支计算出的结果。

class DecoderLayer(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, hidden_dim=16, num_heads=4):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.self_attention = ManualMultiHeadAttention(hidden_dim, num_heads)  # 计算并保存当前步骤的中间状态。
        self.cross_attention = ManualMultiHeadAttention(hidden_dim, num_heads)  # 计算并保存当前步骤的中间状态。
        self.norm1, self.norm2, self.norm3 = (nn.LayerNorm(hidden_dim) for _ in range(3))  # 计算并保存当前步骤的中间状态。
        self.ffn = FeedForward(hidden_dim)  # 计算并保存当前步骤的中间状态。

    def forward(self, target, query_position, memory, memory_position, memory_mask):  # 定义本节可复用的核心函数。
        q = target + query_position  # 计算并保存当前步骤的中间状态。
        target = self.norm1(target + self.self_attention(q, q, target))  # 计算并保存当前步骤的中间状态。
        target = self.norm2(target + self.cross_attention(target + query_position,  # 计算并保存当前步骤的中间状态。
                                                           memory + memory_position, memory,  # 执行当前语句以推进本节示例。
                                                           memory_mask))  # 执行当前语句以推进本节示例。
        return self.norm3(target + self.ffn(target))  # 返回当前分支计算出的结果。

class BoxMLP(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, hidden_dim):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.layers = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),  # 计算并保存当前步骤的中间状态。
                                    nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),  # 执行当前语句以推进本节示例。
                                    nn.Linear(hidden_dim, 4))  # 执行当前语句以推进本节示例。
    def forward(self, x):  # 定义本节可复用的核心函数。
        return self.layers(x).sigmoid()  # 返回当前分支计算出的结果。

class TinyDETR(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_channels=1, hidden_dim=16, num_heads=4,  # 定义本节可复用的核心函数。
                 num_queries=3, num_classes=2):  # 计算并保存当前步骤的中间状态。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.in_channels, self.hidden_dim = int(in_channels), int(hidden_dim)  # 计算并保存当前步骤的中间状态。
        self.num_heads = int(num_heads)  # 计算并保存当前步骤的中间状态。
        self.num_queries, self.num_classes = int(num_queries), int(num_classes)  # 计算并保存当前步骤的中间状态。
        self.backbone = TinyCNNBackbone(in_channels, hidden_dim)  # 计算并保存当前步骤的中间状态。
        self.position = Sine2DPositionEncoding(hidden_dim)  # 计算并保存当前步骤的中间状态。
        self.encoder = EncoderLayer(hidden_dim, num_heads)  # 计算并保存当前步骤的中间状态。
        self.decoder = DecoderLayer(hidden_dim, num_heads)  # 计算并保存当前步骤的中间状态。
        self.query_embedding = nn.Embedding(num_queries, hidden_dim)  # 计算并保存当前步骤的中间状态。
        self.class_head = nn.Linear(hidden_dim, num_classes + 1)  # 计算并保存当前步骤的中间状态。
        self.box_head = BoxMLP(hidden_dim)  # 计算并保存当前步骤的中间状态。

    def forward(self, images, pixel_padding_mask=None):  # 定义本节可复用的核心函数。
        if images.ndim != 4 or images.shape[1] != self.in_channels:  # 按当前条件选择后续控制路径。
            raise ValueError("TinyDETR image shape mismatch")  # 遇到非法合同立即显式失败。
        batch, _, height, width = images.shape  # 计算并保存当前步骤的中间状态。
        if pixel_padding_mask is None:  # 按当前条件选择后续控制路径。
            pixel_padding_mask = torch.zeros(batch, height, width, dtype=torch.bool, device=images.device)  # 计算并保存当前步骤的中间状态。
        if pixel_padding_mask.shape != (batch, height, width) or pixel_padding_mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
            raise ValueError("pixel padding mask must be bool [B,H,W]")  # 遇到非法合同立即显式失败。
        if pixel_padding_mask.all(dim=(1, 2)).any():  # 按当前条件选择后续控制路径。
            raise ValueError("an image cannot be entirely padding")  # 遇到非法合同立即显式失败。
        clean_images = images.masked_fill(pixel_padding_mask[:, None], 0.0)  # 计算并保存当前步骤的中间状态。
        features = self.backbone(clean_images)  # 计算并保存当前步骤的中间状态。
        feature_mask = self.backbone.propagate_padding_mask(pixel_padding_mask)  # 计算并保存当前步骤的中间状态。
        if feature_mask.shape != (batch, *features.shape[-2:]):  # 按当前条件选择后续控制路径。
            raise RuntimeError("mask propagation must match backbone feature shape")  # 遇到非法合同立即显式失败。
        position_2d = self.position(feature_mask)  # 计算并保存当前步骤的中间状态。
        memory = features.flatten(2).transpose(1, 2)  # 计算并保存当前步骤的中间状态。
        memory_position = position_2d.flatten(1, 2)  # 计算并保存当前步骤的中间状态。
        flat_mask = feature_mask.flatten(1)  # 计算并保存当前步骤的中间状态。
        memory = self.encoder(memory, memory_position, flat_mask)  # 计算并保存当前步骤的中间状态。
        query_position = self.query_embedding.weight.unsqueeze(0).expand(batch, -1, -1)  # 计算并保存当前步骤的中间状态。
        target = torch.zeros_like(query_position)  # 计算并保存当前步骤的中间状态。
        decoded = self.decoder(target, query_position, memory, memory_position, flat_mask)  # 计算并保存当前步骤的中间状态。
        return {"pred_logits": self.class_head(decoded), "pred_boxes": self.box_head(decoded)}  # 返回当前分支计算出的结果。

detr_probe = TinyDETR().eval()  # 计算并保存当前步骤的中间状态。
probe_images = torch.randn(2, 1, 16, 16, requires_grad=True)  # 计算并保存当前步骤的中间状态。
probe_predictions = detr_probe(probe_images)  # 计算并保存当前步骤的中间状态。
probe_predictions["pred_logits"].mean().backward()  # 执行当前语句以推进本节示例。
assert probe_predictions["pred_logits"].shape == (2, 3, 3)  # 用受控断言验证关键不变量。
assert probe_predictions["pred_boxes"].shape == (2, 3, 4)  # 用受控断言验证关键不变量。
assert ((probe_predictions["pred_boxes"] > 0) & (probe_predictions["pred_boxes"] < 1)).all()  # 用受控断言验证关键不变量。
assert probe_images.grad is not None and float(probe_images.grad.norm()) > 0  # 用受控断言验证关键不变量。
assert detr_probe.query_embedding.weight.grad is not None  # 用受控断言验证关键不变量。

## 6. padding 不变性与感受野级 mask

若 mask 声明右侧四列是 padding，那么任意修改这些像素都不应改变输出。测试必须在 `eval()` 下进行，排除 BatchNorm 状态变化。输入先清零还不够：三层 `k=3,s=2,p=1` 卷积后，一个 feature token 只要其感受野碰到 padding 就不能作为有效 key。因此 mask 必须逐层使用同参数的 max-pool 保守传播，最终 feature map 是 `2×2`，不能用 adaptive pooling 猜边界。

全 padding 样本没有任何有效 key，attention softmax 无定义，接口应直接拒绝。第 7 列边界 oracle 专门区分真正的感受野传播与错误的 adaptive pooling。

In [ ]:
padding_model = TinyDETR().eval()  # 计算并保存当前步骤的中间状态。
base_image = torch.randn(1, 1, 16, 16)  # 计算并保存当前步骤的中间状态。
pixel_mask = torch.zeros(1, 16, 16, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
pixel_mask[:, :, 12:] = True  # 计算并保存当前步骤的中间状态。
changed_image = base_image.clone()  # 计算并保存当前步骤的中间状态。
changed_image[:, :, :, 12:] = 999.0  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    base_output = padding_model(base_image, pixel_mask)  # 计算并保存当前步骤的中间状态。
    changed_output = padding_model(changed_image, pixel_mask)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(base_output["pred_logits"], changed_output["pred_logits"], atol=1e-6)  # 用受控断言验证关键不变量。
assert torch.allclose(base_output["pred_boxes"], changed_output["pred_boxes"], atol=1e-6)  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    padding_model(base_image, torch.ones_like(pixel_mask))  # 执行当前语句以推进本节示例。
    raise AssertionError("all-padding image must fail")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。

# 第 7 列同时落入最终两个横向 token 的卷积感受野；adaptive pooling 会漏掉右 token。
boundary_mask = torch.zeros(1, 16, 16, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
boundary_mask[:, :, 7] = True  # 计算并保存当前步骤的中间状态。
rf_mask = padding_model.backbone.propagate_padding_mask(boundary_mask)  # 计算并保存当前步骤的中间状态。
adaptive_mask = F.adaptive_max_pool2d(boundary_mask.float().unsqueeze(1), (2, 2)).squeeze(1).bool()  # 计算并保存当前步骤的中间状态。
assert rf_mask.shape == (1, 2, 2)  # 用受控断言验证关键不变量。
assert rf_mask[0, 0].tolist() == [True, True]  # 用受控断言验证关键不变量。
assert adaptive_mask[0, 0].tolist() == [True, False]  # 用受控断言验证关键不变量。
assert not torch.equal(rf_mask, adaptive_mask)  # 用受控断言验证关键不变量。


## 7. box 表示、IoU 与匹配代价

网络输出 normalized `cxcywh`，损失和评估常转换到 `xyxy`。两框 IoU 为

$$\operatorname{IoU}(a,b)=\frac{|a\cap b|}{|a|+|b|-|a\cap b|}.$$

匹配代价组合类别负对数概率、$L_1$ box 距离和 $1-\text{IoU}$。每一项尺度不同，权重属于训练配置，必须进入制品或实验记录。这里的穷举复杂度为 $P(Q,T)=Q!/(Q-T)!$，只适合 $Q,T$ 很小的数值 oracle。

In [ ]:
def validate_detection_target35(target, num_classes):  # 定义本节可复用的核心函数。
    if not isinstance(target, dict) or set(target) != {"labels", "boxes"}:  # 按当前条件选择后续控制路径。
        raise ValueError("target must contain exactly labels and boxes")  # 遇到非法合同立即显式失败。
    labels, boxes = target["labels"], target["boxes"]  # 计算并保存当前步骤的中间状态。
    if labels.dtype != torch.long or labels.ndim != 1 or boxes.shape != (labels.numel(), 4):  # 按当前条件选择后续控制路径。
        raise ValueError("invalid target tensor shape or dtype")  # 遇到非法合同立即显式失败。
    if labels.numel() and ((labels < 0).any() or (labels >= num_classes).any()):  # 按当前条件选择后续控制路径。
        raise ValueError("target labels must be real classes, never the no-object index")  # 遇到非法合同立即显式失败。
    if not torch.isfinite(boxes).all():  # 按当前条件选择后续控制路径。
        raise ValueError("target boxes must be finite")  # 遇到非法合同立即显式失败。
    if boxes.numel():  # 按当前条件选择后续控制路径。
        if (boxes[:, 2:] <= 0).any():  # 按当前条件选择后续控制路径。
            raise ValueError("target width/height must be positive")  # 遇到非法合同立即显式失败。
        xyxy = cxcywh_to_xyxy(boxes)  # 计算并保存当前步骤的中间状态。
        if (xyxy < 0).any() or (xyxy > 1).any():  # 按当前条件选择后续控制路径。
            raise ValueError("target boxes must stay inside normalized image bounds")  # 遇到非法合同立即显式失败。

def cxcywh_to_xyxy(boxes):  # 定义本节可复用的核心函数。
    if boxes.shape[-1] != 4:  # 按当前条件选择后续控制路径。
        raise ValueError("boxes must end in four coordinates")  # 遇到非法合同立即显式失败。
    center, size = boxes[..., :2], boxes[..., 2:]  # 计算并保存当前步骤的中间状态。
    return torch.cat([center - size / 2, center + size / 2], dim=-1)  # 返回当前分支计算出的结果。

def pairwise_iou(boxes1, boxes2):  # 定义本节可复用的核心函数。
    if boxes1.ndim != 2 or boxes2.ndim != 2:  # 按当前条件选择后续控制路径。
        raise ValueError("pairwise_iou expects two matrices")  # 遇到非法合同立即显式失败。
    top_left = torch.maximum(boxes1[:, None, :2], boxes2[None, :, :2])  # 计算并保存当前步骤的中间状态。
    bottom_right = torch.minimum(boxes1[:, None, 2:], boxes2[None, :, 2:])  # 计算并保存当前步骤的中间状态。
    intersection = (bottom_right - top_left).clamp_min(0).prod(dim=-1)  # 计算并保存当前步骤的中间状态。
    area1 = (boxes1[:, 2:] - boxes1[:, :2]).clamp_min(0).prod(dim=-1)  # 计算并保存当前步骤的中间状态。
    area2 = (boxes2[:, 2:] - boxes2[:, :2]).clamp_min(0).prod(dim=-1)  # 计算并保存当前步骤的中间状态。
    union = area1[:, None] + area2[None, :] - intersection  # 计算并保存当前步骤的中间状态。
    return intersection / union.clamp_min(1e-8)  # 返回当前分支计算出的结果。

def optimal_bipartite_match(pred_logits, pred_boxes, target_labels, target_boxes,  # 定义本节可复用的核心函数。
                            class_weight=1.0, l1_weight=3.0, iou_weight=2.0):  # 计算并保存当前步骤的中间状态。
    queries = pred_logits.shape[0]  # 计算并保存当前步骤的中间状态。
    targets = target_labels.numel()  # 计算并保存当前步骤的中间状态。
    if (pred_logits.ndim != 2 or pred_logits.shape[1] < 2 or  # 按当前条件选择后续控制路径。
            pred_boxes.shape != (queries, 4) or target_boxes.shape != (targets, 4)):  # 计算并保存当前步骤的中间状态。
        raise ValueError("matching tensor shape mismatch")  # 遇到非法合同立即显式失败。
    validate_detection_target35({"labels": target_labels, "boxes": target_boxes},  # 执行当前语句以推进本节示例。
                                pred_logits.shape[1] - 1)  # 执行当前语句以推进本节示例。
    if not all(math.isfinite(value) and value >= 0  # 按当前条件选择后续控制路径。
               for value in (class_weight, l1_weight, iou_weight)):  # 遍历输入元素以累积或检查结果。
        raise ValueError("matching weights must be finite and non-negative")  # 遇到非法合同立即显式失败。
    if targets == 0:  # 按当前条件选择后续控制路径。
        empty = torch.empty(0, dtype=torch.long, device=pred_logits.device)  # 计算并保存当前步骤的中间状态。
        return empty, empty  # 返回当前分支计算出的结果。
    if targets > queries:  # 按当前条件选择后续控制路径。
        raise ValueError("targets exceed available object queries")  # 遇到非法合同立即显式失败。
    class_cost = -pred_logits.log_softmax(-1)[:, target_labels]  # 计算并保存当前步骤的中间状态。
    l1_cost = torch.cdist(pred_boxes, target_boxes, p=1)  # 计算并保存当前步骤的中间状态。
    iou_cost = 1 - pairwise_iou(cxcywh_to_xyxy(pred_boxes), cxcywh_to_xyxy(target_boxes))  # 计算并保存当前步骤的中间状态。
    cost = class_weight * class_cost + l1_weight * l1_cost + iou_weight * iou_cost  # 计算并保存当前步骤的中间状态。
    best_assignment, best_value = None, float("inf")  # 计算并保存当前步骤的中间状态。
    for assignment in permutations(range(queries), targets):  # 遍历输入元素以累积或检查结果。
        value = float(sum(cost[assignment[t], t].detach() for t in range(targets)))  # 计算并保存当前步骤的中间状态。
        if value < best_value:  # 按当前条件选择后续控制路径。
            best_value, best_assignment = value, assignment  # 计算并保存当前步骤的中间状态。
    return (torch.tensor(best_assignment, device=pred_logits.device),  # 返回当前分支计算出的结果。
            torch.arange(targets, device=pred_logits.device))  # 计算并保存当前步骤的中间状态。

# 两个完全命中的 box 应得到单位 IoU；不相交得到 0。
iou_oracle = pairwise_iou(torch.tensor([[0., 0., 1., 1.], [0., 0., .2, .2]]),  # 计算并保存当前步骤的中间状态。
                          torch.tensor([[0., 0., 1., 1.], [.8, .8, 1., 1.]]))  # 执行当前语句以推进本节示例。
assert torch.allclose(iou_oracle.diag(), torch.tensor([1., 0.]))  # 用受控断言验证关键不变量。
match_logits = torch.tensor([[6., 0., -2.], [0., 6., -2.], [-2., -2., 6.]])  # 计算并保存当前步骤的中间状态。
match_boxes = torch.tensor([[.2, .2, .2, .2], [.8, .8, .2, .2], [.5, .5, .1, .1]])  # 计算并保存当前步骤的中间状态。
matched_q, matched_t = optimal_bipartite_match(match_logits, match_boxes,  # 计算并保存当前步骤的中间状态。
                                                torch.tensor([1, 0]),  # 执行当前语句以推进本节示例。
                                                torch.tensor([[.8, .8, .2, .2], [.2, .2, .2, .2]]))  # 执行当前语句以推进本节示例。
assert matched_q.tolist() == [1, 0] and matched_t.tolist() == [0, 1]  # 用受控断言验证关键不变量。

## 8. set loss、no-object 与排列不变性

匹配后，所有 query 都进入 classification loss：匹配 query 使用真实类别，未匹配 query 使用 no-object。由于空类别数量通常很多，给 no-object 较小权重 `eos_coef`。只有匹配 query 计算 box $L_1$ 和 IoU loss；空目标时 box loss 应为可反传的精确零。

对预测 query 任意置换，总损失必须不变。这项 metamorphic test 能发现“按下标和 target 对齐”的伪 DETR 实现。

In [ ]:
def set_loss_single(pred_logits, pred_boxes, target, num_classes=2,  # 定义本节可复用的核心函数。
                    eos_coef=0.2, bbox_weight=3.0, iou_weight=2.0):  # 计算并保存当前步骤的中间状态。
    if (pred_logits.ndim != 2 or pred_logits.shape[1] != num_classes + 1 or  # 按当前条件选择后续控制路径。
            pred_boxes.shape != (pred_logits.shape[0], 4) or pred_logits.shape[0] < 1):  # 计算并保存当前步骤的中间状态。
        raise ValueError("prediction shape mismatch")  # 遇到非法合同立即显式失败。
    if not (torch.isfinite(pred_logits).all() and torch.isfinite(pred_boxes).all()):  # 按当前条件选择后续控制路径。
        raise ValueError("predictions must be finite")  # 遇到非法合同立即显式失败。
    if not math.isfinite(eos_coef) or eos_coef <= 0:  # 按当前条件选择后续控制路径。
        raise ValueError("eos_coef must be finite and positive")  # 遇到非法合同立即显式失败。
    validate_detection_target35(target, num_classes)  # 执行当前语句以推进本节示例。
    labels, boxes = target["labels"], target["boxes"]  # 计算并保存当前步骤的中间状态。
    matched_q, matched_t = optimal_bipartite_match(pred_logits, pred_boxes, labels, boxes)  # 计算并保存当前步骤的中间状态。
    class_targets = torch.full((pred_logits.shape[0],), num_classes,  # 计算并保存当前步骤的中间状态。
                               dtype=torch.long, device=pred_logits.device)  # 计算并保存当前步骤的中间状态。
    if matched_q.numel():  # 按当前条件选择后续控制路径。
        class_targets[matched_q] = labels[matched_t]  # 计算并保存当前步骤的中间状态。
    per_query_ce = F.cross_entropy(pred_logits, class_targets, reduction="none")  # 计算并保存当前步骤的中间状态。
    query_weights = torch.ones_like(per_query_ce)  # 计算并保存当前步骤的中间状态。
    query_weights[class_targets == num_classes] = eos_coef  # 计算并保存当前步骤的中间状态。
    # 固定 query 数作 denominator；不再被 F.cross_entropy(weight=..., mean) 的权重和抵消。
    class_loss = (per_query_ce * query_weights).sum() / pred_logits.shape[0]  # 计算并保存当前步骤的中间状态。
    if matched_q.numel():  # 按当前条件选择后续控制路径。
        selected_pred, selected_target = pred_boxes[matched_q], boxes[matched_t]  # 计算并保存当前步骤的中间状态。
        l1_loss = F.l1_loss(selected_pred, selected_target)  # 计算并保存当前步骤的中间状态。
        iou_matrix = pairwise_iou(cxcywh_to_xyxy(selected_pred), cxcywh_to_xyxy(selected_target))  # 计算并保存当前步骤的中间状态。
        iou_loss = 1 - iou_matrix.diag().mean()  # 计算并保存当前步骤的中间状态。
    else:  # 处理前置条件不成立的分支。
        l1_loss = pred_boxes.sum() * 0.0  # 计算并保存当前步骤的中间状态。
        iou_loss = pred_boxes.sum() * 0.0  # 计算并保存当前步骤的中间状态。
    total = class_loss + bbox_weight * l1_loss + iou_weight * iou_loss  # 计算并保存当前步骤的中间状态。
    return total, {"class": class_loss, "l1": l1_loss, "iou": iou_loss}  # 返回当前分支计算出的结果。

def batch_set_loss(outputs, targets):  # 定义本节可复用的核心函数。
    losses = [set_loss_single(outputs["pred_logits"][i], outputs["pred_boxes"][i], target)  # 计算并保存当前步骤的中间状态。
              for i, target in enumerate(targets)]  # 遍历输入元素以累积或检查结果。
    return torch.stack([item[0] for item in losses]).mean()  # 返回当前分支计算出的结果。

random_logits = torch.randn(4, 3)  # 计算并保存当前步骤的中间状态。
random_boxes = torch.rand(4, 4)  # 计算并保存当前步骤的中间状态。
two_targets = {"labels": torch.tensor([0, 1]),  # 计算并保存当前步骤的中间状态。
               "boxes": torch.tensor([[.2, .3, .2, .3], [.7, .6, .25, .2]])}  # 执行当前语句以推进本节示例。
loss_original = set_loss_single(random_logits, random_boxes, two_targets)[0]  # 计算并保存当前步骤的中间状态。
query_permutation = torch.tensor([2, 0, 3, 1])  # 计算并保存当前步骤的中间状态。
loss_permuted = set_loss_single(random_logits[query_permutation],  # 计算并保存当前步骤的中间状态。
                                random_boxes[query_permutation], two_targets)[0]  # 执行当前语句以推进本节示例。
assert torch.allclose(loss_original, loss_permuted, atol=1e-6)  # 用受控断言验证关键不变量。

empty_target = {"labels": torch.empty(0, dtype=torch.long), "boxes": torch.empty(0, 4)}  # 计算并保存当前步骤的中间状态。
empty_total, empty_parts = set_loss_single(random_logits, random_boxes, empty_target)  # 计算并保存当前步骤的中间状态。
empty_low = set_loss_single(random_logits, random_boxes, empty_target, eos_coef=0.1)[0]  # 计算并保存当前步骤的中间状态。
empty_high = set_loss_single(random_logits, random_boxes, empty_target, eos_coef=0.5)[0]  # 计算并保存当前步骤的中间状态。
assert empty_total > 0 and empty_parts["class"] > 0  # 用受控断言验证关键不变量。
assert empty_parts["l1"].item() == 0.0 and empty_parts["iou"].item() == 0.0  # 用受控断言验证关键不变量。
assert torch.allclose(empty_high, 5.0 * empty_low, atol=1e-6)  # eos 权重对空图真实生效

invalid_targets = [  # 计算并保存当前步骤的中间状态。
    {"labels": torch.tensor([2]), "boxes": torch.tensor([[.5, .5, .2, .2]])},  # 执行当前语句以推进本节示例。
    {"labels": torch.tensor([0]), "boxes": torch.tensor([[.5, .5, 0., .2]])},  # 执行当前语句以推进本节示例。
    {"labels": torch.tensor([0]), "boxes": torch.tensor([[.95, .5, .2, .2]])},  # 执行当前语句以推进本节示例。
    {"labels": torch.tensor([0]), "boxes": torch.tensor([[float("nan"), .5, .2, .2]])},  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
for invalid_target in invalid_targets:  # 遍历输入元素以累积或检查结果。
    try:  # 尝试执行可能失败的受控操作。
        set_loss_single(random_logits, random_boxes, invalid_target)  # 执行当前语句以推进本节示例。
        raise AssertionError("invalid target must fail")  # 遇到非法合同立即显式失败。
    except ValueError:  # 捕获预期异常并验证失败分支。
        pass  # 调整当前循环或占位控制流。

## 9. 离线合成检测数据

每张 $16\times16$ 图像含一个矩形：类别 0 是横向矩形，类别 1 是纵向矩形；中心位置与噪声变化。target 保持为每图一个字典，box 是相对尺寸的 `cxcywh`。

真实检测 batch 有不同图像大小和不同目标数，需要 pad image 并生成 pixel mask；本数据固定尺寸是为了隔离 matching 与 box 学习。前面的测试已经单独覆盖 padding 行为，后面的空目标测试覆盖零目标分支。

In [ ]:
def make_detection_batch(count, seed):  # 定义本节可复用的核心函数。
    generator = torch.Generator().manual_seed(seed)  # 计算并保存当前步骤的中间状态。
    images, targets = torch.zeros(count, 1, 16, 16), []  # 计算并保存当前步骤的中间状态。
    for index in range(count):  # 遍历输入元素以累积或检查结果。
        label = index % 2  # 计算并保存当前步骤的中间状态。
        box_width, box_height = ((6, 3) if label == 0 else (3, 6))  # 计算并保存当前步骤的中间状态。
        # 三个受控位置在各 split 重复出现，噪声随 seed 改变；便于快速验证定位计算图。
        left = (2, 6, 9)[(index // 2) % 3]  # 计算并保存当前步骤的中间状态。
        top = (2, 7, 9)[(index // 6) % 3]  # 计算并保存当前步骤的中间状态。
        left = min(left, 15 - box_width)  # 计算并保存当前步骤的中间状态。
        top = min(top, 15 - box_height)  # 计算并保存当前步骤的中间状态。
        images[index, 0, top:top + box_height, left:left + box_width] = 1.0  # 计算并保存当前步骤的中间状态。
        images[index] += 0.03 * torch.randn(images[index].shape, generator=generator)  # 计算并保存当前步骤的中间状态。
        cx = (left + box_width / 2) / 16  # 计算并保存当前步骤的中间状态。
        cy = (top + box_height / 2) / 16  # 计算并保存当前步骤的中间状态。
        targets.append({"labels": torch.tensor([label], dtype=torch.long),  # 计算并保存当前步骤的中间状态。
                        "boxes": torch.tensor([[cx, cy, box_width / 16, box_height / 16]],  # 执行当前语句以推进本节示例。
                                              dtype=torch.float32)})  # 计算并保存当前步骤的中间状态。
    return images, targets  # 返回当前分支计算出的结果。

train_images, train_targets = make_detection_batch(12, SEED + 1)  # 计算并保存当前步骤的中间状态。
valid_images, valid_targets = make_detection_batch(12, SEED + 2)  # 计算并保存当前步骤的中间状态。
test_images, test_targets = make_detection_batch(12, SEED + 3)  # 计算并保存当前步骤的中间状态。
assert train_images.shape == (12, 1, 16, 16)  # 用受控断言验证关键不变量。
assert all(target["boxes"].shape == (1, 4) for target in train_targets)  # 用受控断言验证关键不变量。
assert all(((target["boxes"] >= 0) & (target["boxes"] <= 1)).all() for target in train_targets)  # 用受控断言验证关键不变量。
assert {int(target["labels"][0]) for target in train_targets} == {0, 1}  # 用受控断言验证关键不变量。

## 10. 受控训练与 validation checkpoint

训练同时更新 CNN、position-aware encoder/decoder、object queries 和两个 heads。每 10 步仅用 validation 选择 checkpoint；test 不参与选择。由于 matching 离散，匹配索引不求梯度，但选中的 logits/box loss 对网络保持可微。

第一步检查 backbone、query embedding、class head 和 box head 都有有限非零梯度。训练损失下降只说明实现可学习；数据极简，不能拿这里的数值同 COCO 指标比较。

In [ ]:
torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
model35 = TinyDETR().to(DEVICE)  # 计算并保存当前步骤的中间状态。
optimizer = torch.optim.Adam(model35.parameters(), lr=0.015)  # 计算并保存当前步骤的中间状态。
history, best_validation, best_state = [], float("inf"), None  # 计算并保存当前步骤的中间状态。

for step in range(41):  # 遍历输入元素以累积或检查结果。
    model35.train()  # 执行当前语句以推进本节示例。
    optimizer.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
    predictions = model35(train_images)  # 计算并保存当前步骤的中间状态。
    loss = batch_set_loss(predictions, train_targets)  # 计算并保存当前步骤的中间状态。
    loss.backward()  # 执行当前语句以推进本节示例。
    if step == 0:  # 按当前条件选择后续控制路径。
        gradient_checks = {  # 计算并保存当前步骤的中间状态。
            "backbone": model35.backbone.network[0].weight.grad.norm(),  # 执行当前语句以推进本节示例。
            "queries": model35.query_embedding.weight.grad.norm(),  # 执行当前语句以推进本节示例。
            "class_head": model35.class_head.weight.grad.norm(),  # 执行当前语句以推进本节示例。
            "box_head": model35.box_head.layers[-1].weight.grad.norm(),  # 执行当前语句以推进本节示例。
        }  # 执行当前语句以推进本节示例。
    torch.nn.utils.clip_grad_norm_(model35.parameters(), 2.0)  # 执行当前语句以推进本节示例。
    optimizer.step()  # 执行当前语句以推进本节示例。
    history.append(float(loss.detach()))  # 执行当前语句以推进本节示例。
    if step % 10 == 0:  # 按当前条件选择后续控制路径。
        model35.eval()  # 执行当前语句以推进本节示例。
        with torch.no_grad():  # 在受管理的上下文中执行操作。
            validation_loss = float(batch_set_loss(model35(valid_images), valid_targets))  # 计算并保存当前步骤的中间状态。
        if validation_loss < best_validation:  # 按当前条件选择后续控制路径。
            best_validation, best_state = validation_loss, deepcopy(model35.state_dict())  # 计算并保存当前步骤的中间状态。

assert best_state is not None  # 用受控断言验证关键不变量。
model35.load_state_dict(best_state)  # 执行当前语句以推进本节示例。
assert all(torch.isfinite(value) and float(value) > 0 for value in gradient_checks.values())  # 用受控断言验证关键不变量。
assert sum(history[-10:]) / 10 < history[0] * 0.75  # 用受控断言验证关键不变量。
print({"train_loss_first_last": [history[0], history[-1]],  # 执行当前语句以推进本节示例。
       "best_validation_set_loss": best_validation})  # 执行当前语句以推进本节示例。

## 11. 推理：去掉 no-object、clip box，再评估

推理不能对包含 no-object 的全部类别直接 `argmax` 后当目标返回。先取 softmax，保留最佳真实类别及其分数，再按阈值过滤；box 从 cxcywh 转 xyxy 并 clip 到 `[0,1]`。同一 query 只给一个预测，DETR 通常不依赖 NMS。

本例报告每图最高分预测的类别准确率和 IoU，仅作为受控 smoke metric。真实检测必须实现按类别 AP、不同 IoU threshold 的 mAP、small/medium/large 分桶、空图 false positives 与置信度校准。

In [ ]:
@torch.no_grad()  # 为下方定义附加声明式配置。
def postprocess_detr(outputs, score_threshold=0.25):  # 定义本节可复用的核心函数。
    probabilities = outputs["pred_logits"].softmax(-1)  # 计算并保存当前步骤的中间状态。
    real_scores, real_labels = probabilities[..., :-1].max(-1)  # 计算并保存当前步骤的中间状态。
    no_object_scores = probabilities[..., -1]  # 计算并保存当前步骤的中间状态。
    boxes_xyxy = cxcywh_to_xyxy(outputs["pred_boxes"]).clamp(0.0, 1.0)  # 计算并保存当前步骤的中间状态。
    results = []  # 计算并保存当前步骤的中间状态。
    for sample in range(probabilities.shape[0]):  # 遍历输入元素以累积或检查结果。
        keep = (real_scores[sample] >= score_threshold) & (real_scores[sample] > no_object_scores[sample])  # 计算并保存当前步骤的中间状态。
        results.append({"scores": real_scores[sample, keep],  # 执行当前语句以推进本节示例。
                        "labels": real_labels[sample, keep],  # 执行当前语句以推进本节示例。
                        "boxes": boxes_xyxy[sample, keep]})  # 执行当前语句以推进本节示例。
    return results  # 返回当前分支计算出的结果。

model35.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    test_outputs = model35(test_images)  # 计算并保存当前步骤的中间状态。
test_results = postprocess_detr(test_outputs, score_threshold=0.20)  # 计算并保存当前步骤的中间状态。
best_ious, correct_classes = [], []  # 计算并保存当前步骤的中间状态。
for result, target in zip(test_results, test_targets):  # 遍历输入元素以累积或检查结果。
    assert result["boxes"].ndim == 2 and result["boxes"].shape[-1] == 4  # 用受控断言验证关键不变量。
    assert ((result["boxes"] >= 0) & (result["boxes"] <= 1)).all()  # 用受控断言验证关键不变量。
    if len(result["scores"]) == 0:  # 按当前条件选择后续控制路径。
        best_ious.append(0.0); correct_classes.append(0.0)  # 执行当前语句以推进本节示例。
        continue  # 调整当前循环或占位控制流。
    best = int(result["scores"].argmax())  # 计算并保存当前步骤的中间状态。
    target_xyxy = cxcywh_to_xyxy(target["boxes"])  # 计算并保存当前步骤的中间状态。
    best_ious.append(float(pairwise_iou(result["boxes"][best:best+1], target_xyxy)[0, 0]))  # 执行当前语句以推进本节示例。
    correct_classes.append(float(result["labels"][best] == target["labels"][0]))  # 计算并保存当前步骤的中间状态。

mean_iou = sum(best_ious) / len(best_ious)  # 计算并保存当前步骤的中间状态。
class_accuracy = sum(correct_classes) / len(correct_classes)  # 计算并保存当前步骤的中间状态。
mean_train_box = torch.cat([target["boxes"] for target in train_targets]).mean(0, keepdim=True)  # 计算并保存当前步骤的中间状态。
constant_box_iou = sum(  # 计算并保存当前步骤的中间状态。
    float(pairwise_iou(cxcywh_to_xyxy(mean_train_box),  # 执行当前语句以推进本节示例。
                       cxcywh_to_xyxy(target["boxes"]))[0, 0])  # 执行当前语句以推进本节示例。
    for target in test_targets  # 遍历输入元素以累积或检查结果。
) / len(test_targets)  # 执行当前语句以推进本节示例。
print({"controlled_test_mean_best_iou": mean_iou,  # 执行当前语句以推进本节示例。
       "controlled_test_class_accuracy": class_accuracy,  # 执行当前语句以推进本节示例。
       "constant_box_baseline_iou": constant_box_iou,  # 执行当前语句以推进本节示例。
       "detections_per_image": [len(item["scores"]) for item in test_results]})  # 执行当前语句以推进本节示例。
assert mean_iou >= constant_box_iou + 0.03  # 用受控断言验证关键不变量。
assert class_accuracy >= 0.80  # 用受控断言验证关键不变量。

## 12. 检测制品：外部信任锚、数据快照与语义合同

“manifest 里放一个 hash”只能发现传输损坏，不能阻止攻击者整体替换模型后把内部 hash 全部重签。这里把信任边界改成两层：调用方 package 仍携带内部摘要，但 loader 还必须命中发布者侧只读登记表 `artifact_id/version -> expected bundle digest`。bundle 摘要对 canonical manifest 与逐 tensor 的 `key/dtype/shape/bytes` 一起做长度分隔哈希；登记表不来自 package。

检测语义也必须整体绑定：模型 config、类别顺序、`normalized_cxcywh`、CNN 感受野 mask 传播、score threshold、matching/loss 权重，以及 train/validation/test 的图像、label、box 与 split seed。加载器会重建本受控数据并复核 snapshot。生产中这个只读登记表应由签名发布元数据、透明日志或只读制品服务提供，而不是和模型放在同一个可替换目录。

In [ ]:
def canonical_json35(value):  # 定义本节可复用的核心函数。
    return json.dumps(value, sort_keys=True, separators=(",", ":"),  # 返回当前分支计算出的结果。
                      ensure_ascii=False).encode("utf-8")  # 计算并保存当前步骤的中间状态。

def _feed_digest35(hasher, payload):  # 定义本节可复用的核心函数。
    hasher.update(len(payload).to_bytes(8, "big"))  # 执行当前语句以推进本节示例。
    hasher.update(payload)  # 执行当前语句以推进本节示例。

def clone_state35(state_dict):  # 定义本节可复用的核心函数。
    if not hasattr(state_dict, "items"):  # 按当前条件选择后续控制路径。
        raise ValueError("state_dict must be a mapping")  # 遇到非法合同立即显式失败。
    cloned = {}  # 计算并保存当前步骤的中间状态。
    for key, tensor in state_dict.items():  # 遍历输入元素以累积或检查结果。
        if not isinstance(key, str) or not isinstance(tensor, torch.Tensor):  # 按当前条件选择后续控制路径。
            raise ValueError("state entries must be string -> Tensor")  # 遇到非法合同立即显式失败。
        cloned[key] = tensor.detach().cpu().contiguous().clone()  # 计算并保存当前步骤的中间状态。
    return cloned  # 返回当前分支计算出的结果。

def _update_state_digest35(hasher, state_dict):  # 定义本节可复用的核心函数。
    if not isinstance(state_dict, dict) or not state_dict:  # 按当前条件选择后续控制路径。
        raise ValueError("artifact state_dict must be a non-empty plain dict")  # 遇到非法合同立即显式失败。
    for key in sorted(state_dict):  # 遍历输入元素以累积或检查结果。
        tensor = state_dict[key]  # 计算并保存当前步骤的中间状态。
        if not isinstance(key, str) or not isinstance(tensor, torch.Tensor):  # 按当前条件选择后续控制路径。
            raise ValueError("invalid state entry")  # 遇到非法合同立即显式失败。
        cpu = tensor.detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        header = canonical_json35({"key": key, "dtype": str(cpu.dtype),  # 计算并保存当前步骤的中间状态。
                                   "shape": list(cpu.shape)})  # 执行当前语句以推进本节示例。
        raw = cpu.reshape(-1).view(torch.uint8).numpy().tobytes()  # 计算并保存当前步骤的中间状态。
        _feed_digest35(hasher, header)  # 执行当前语句以推进本节示例。
        _feed_digest35(hasher, raw)  # 执行当前语句以推进本节示例。

def canonical_state_digest35(state_dict):  # 定义本节可复用的核心函数。
    hasher = sha256()  # 计算并保存当前步骤的中间状态。
    _feed_digest35(hasher, b"canonical-state-dict-v1")  # 执行当前语句以推进本节示例。
    _update_state_digest35(hasher, state_dict)  # 执行当前语句以推进本节示例。
    return hasher.hexdigest()  # 返回当前分支计算出的结果。

def canonical_bundle_digest35(manifest, state_dict):  # 定义本节可复用的核心函数。
    hasher = sha256()  # 计算并保存当前步骤的中间状态。
    _feed_digest35(hasher, b"canonical-model-bundle-v1")  # 执行当前语句以推进本节示例。
    _feed_digest35(hasher, canonical_json35(manifest))  # 执行当前语句以推进本节示例。
    _update_state_digest35(hasher, state_dict)  # 执行当前语句以推进本节示例。
    return hasher.hexdigest()  # 返回当前分支计算出的结果。

def state_schema35(state_dict):  # 定义本节可复用的核心函数。
    return [{"key": key, "dtype": str(state_dict[key].dtype),  # 返回当前分支计算出的结果。
             "shape": list(state_dict[key].shape)} for key in sorted(state_dict)]  # 执行当前语句以推进本节示例。

def detection_split_digest35(images, targets):  # 定义本节可复用的核心函数。
    tensors = {"images": images}  # 计算并保存当前步骤的中间状态。
    for index, target in enumerate(targets):  # 遍历输入元素以累积或检查结果。
        tensors[f"target/{index}/labels"] = target["labels"]  # 计算并保存当前步骤的中间状态。
        tensors[f"target/{index}/boxes"] = target["boxes"]  # 计算并保存当前步骤的中间状态。
    return canonical_state_digest35(clone_state35(tensors))  # 返回当前分支计算出的结果。

def expected_data_contract35():  # 定义本节可复用的核心函数。
    split_specs = {"train": (12, SEED + 1), "validation": (12, SEED + 2),  # 计算并保存当前步骤的中间状态。
                   "test": (12, SEED + 3)}  # 执行当前语句以推进本节示例。
    splits = {}  # 计算并保存当前步骤的中间状态。
    for name, (count, seed) in split_specs.items():  # 遍历输入元素以累积或检查结果。
        images, targets = make_detection_batch(count, seed)  # 计算并保存当前步骤的中间状态。
        splits[name] = {"count": count, "seed": seed,  # 计算并保存当前步骤的中间状态。
                        "images_targets_sha256": detection_split_digest35(images, targets)}  # 执行当前语句以推进本节示例。
    return {"dataset_recipe": "controlled-rectangles-v1", "splits": splits,  # 返回当前分支计算出的结果。
            "target_schema": {"labels": "int64[N]", "boxes": "float32[N,4]"}}  # 执行当前语句以推进本节示例。

EXPECTED_CONFIG35 = {"in_channels": 1, "hidden_dim": 16, "num_heads": 4,  # 计算并保存当前步骤的中间状态。
                     "num_queries": 3, "num_classes": 2}  # 执行当前语句以推进本节示例。
EXPECTED_CLASSES35 = ["horizontal", "vertical"]  # 计算并保存当前步骤的中间状态。
EXPECTED_PREPROCESS35 = {  # 计算并保存当前步骤的中间状态。
    "input_shape": [1, 16, 16], "dtype": "float32", "normalization": "none",  # 执行当前语句以推进本节示例。
    "padding_mask_true_means": "padding",  # 执行当前语句以推进本节示例。
    "padding_mask_downsample": "conv-receptive-field-max-pool-k3-s2-p1-x3/v1",  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
EXPECTED_MATCHING35 = {"algorithm": "exact-global-exhaustive-oracle",  # 计算并保存当前步骤的中间状态。
                       "class_weight": 1.0, "l1_weight": 3.0, "iou_weight": 2.0}  # 执行当前语句以推进本节示例。
EXPECTED_LOSS35 = {"classification": "weighted-ce-fixed-query-denominator",  # 计算并保存当前步骤的中间状态。
                   "eos_coef": 0.2, "bbox_weight": 3.0, "iou_weight": 2.0}  # 执行当前语句以推进本节示例。
ARTIFACT_ID35, ARTIFACT_VERSION35 = "vision.controlled-detr", "1.0.0"  # 计算并保存当前步骤的中间状态。

def make_detr_artifact(model):  # 定义本节可复用的核心函数。
    if type(model) is not TinyDETR:  # 按当前条件选择后续控制路径。
        raise ValueError("publisher only accepts the audited TinyDETR class")  # 遇到非法合同立即显式失败。
    config = {"in_channels": model.in_channels, "hidden_dim": model.hidden_dim,  # 计算并保存当前步骤的中间状态。
              "num_heads": model.num_heads, "num_queries": model.num_queries,  # 执行当前语句以推进本节示例。
              "num_classes": model.num_classes}  # 执行当前语句以推进本节示例。
    state = clone_state35(model.state_dict())  # 计算并保存当前步骤的中间状态。
    manifest = {  # 计算并保存当前步骤的中间状态。
        "schema_version": 2, "artifact_id": ARTIFACT_ID35,  # 执行当前语句以推进本节示例。
        "artifact_version": ARTIFACT_VERSION35, "architecture": "TinyDETR",  # 执行当前语句以推进本节示例。
        "config": config, "model_state_schema": state_schema35(state),  # 执行当前语句以推进本节示例。
        "class_names": list(EXPECTED_CLASSES35),  # 执行当前语句以推进本节示例。
        "preprocess": deepcopy(EXPECTED_PREPROCESS35),  # 执行当前语句以推进本节示例。
        "box_format": "normalized_cxcywh", "score_threshold": 0.20,  # 执行当前语句以推进本节示例。
        "matching": deepcopy(EXPECTED_MATCHING35), "loss": deepcopy(EXPECTED_LOSS35),  # 执行当前语句以推进本节示例。
        "data_contract": expected_data_contract35(),  # 执行当前语句以推进本节示例。
        "state_digest_sha256": canonical_state_digest35(state),  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。
    return {"manifest": manifest,  # 返回当前分支计算出的结果。
            "manifest_sha256": sha256(canonical_json35(manifest)).hexdigest(),  # 执行当前语句以推进本节示例。
            "bundle_sha256": canonical_bundle_digest35(manifest, state),  # 执行当前语句以推进本节示例。
            "state_dict": state}  # 执行当前语句以推进本节示例。

def validate_detr_contract35(manifest, state_dict):  # 定义本节可复用的核心函数。
    required = {"schema_version", "artifact_id", "artifact_version", "architecture",  # 计算并保存当前步骤的中间状态。
                "config", "model_state_schema", "class_names", "preprocess",  # 执行当前语句以推进本节示例。
                "box_format", "score_threshold", "matching", "loss",  # 执行当前语句以推进本节示例。
                "data_contract", "state_digest_sha256"}  # 执行当前语句以推进本节示例。
    if set(manifest) != required or manifest["schema_version"] != 2:  # 按当前条件选择后续控制路径。
        raise ValueError("manifest schema mismatch")  # 遇到非法合同立即显式失败。
    if (manifest["artifact_id"], manifest["artifact_version"]) != (ARTIFACT_ID35, ARTIFACT_VERSION35):  # 按当前条件选择后续控制路径。
        raise ValueError("artifact identity mismatch")  # 遇到非法合同立即显式失败。
    if manifest["architecture"] != "TinyDETR" or manifest["config"] != EXPECTED_CONFIG35:  # 按当前条件选择后续控制路径。
        raise ValueError("model config mismatch")  # 遇到非法合同立即显式失败。
    names = manifest["class_names"]  # 计算并保存当前步骤的中间状态。
    if (names != EXPECTED_CLASSES35 or len(names) != EXPECTED_CONFIG35["num_classes"] or  # 按当前条件选择后续控制路径。
            len(set(names)) != len(names) or any(not isinstance(x, str) or not x.strip() for x in names)):  # 计算并保存当前步骤的中间状态。
        raise ValueError("class mapping mismatch")  # 遇到非法合同立即显式失败。
    if manifest["preprocess"] != EXPECTED_PREPROCESS35:  # 按当前条件选择后续控制路径。
        raise ValueError("preprocess contract mismatch")  # 遇到非法合同立即显式失败。
    if manifest["box_format"] != "normalized_cxcywh" or manifest["score_threshold"] != 0.20:  # 按当前条件选择后续控制路径。
        raise ValueError("postprocess contract mismatch")  # 遇到非法合同立即显式失败。
    if manifest["matching"] != EXPECTED_MATCHING35 or manifest["loss"] != EXPECTED_LOSS35:  # 按当前条件选择后续控制路径。
        raise ValueError("matching/loss contract mismatch")  # 遇到非法合同立即显式失败。
    if manifest["data_contract"] != expected_data_contract35():  # 按当前条件选择后续控制路径。
        raise ValueError("image/target/split snapshot mismatch")  # 遇到非法合同立即显式失败。
    expected_schema = state_schema35(TinyDETR(**EXPECTED_CONFIG35).state_dict())  # 计算并保存当前步骤的中间状态。
    if manifest["model_state_schema"] != expected_schema or state_schema35(state_dict) != expected_schema:  # 按当前条件选择后续控制路径。
        raise ValueError("model state schema mismatch")  # 遇到非法合同立即显式失败。

def load_trusted_detr(artifact):  # 定义本节可复用的核心函数。
    if not isinstance(artifact, dict) or set(artifact) != {  # 按当前条件选择后续控制路径。
            "manifest", "manifest_sha256", "bundle_sha256", "state_dict"}:  # 执行当前语句以推进本节示例。
        raise ValueError("artifact package schema mismatch")  # 遇到非法合同立即显式失败。
    manifest, state = artifact["manifest"], artifact["state_dict"]  # 计算并保存当前步骤的中间状态。
    if not isinstance(manifest, dict):  # 按当前条件选择后续控制路径。
        raise ValueError("manifest must be a dict")  # 遇到非法合同立即显式失败。
    actual_bundle = canonical_bundle_digest35(manifest, state)  # 计算并保存当前步骤的中间状态。
    identity = (manifest.get("artifact_id"), manifest.get("artifact_version"))  # 计算并保存当前步骤的中间状态。
    expected_bundle = PUBLISHER_REGISTRY35.get(identity)  # 计算并保存当前步骤的中间状态。
    if expected_bundle is None or actual_bundle != expected_bundle:  # 按当前条件选择后续控制路径。
        raise ValueError("publisher registry rejected this bundle")  # 遇到非法合同立即显式失败。
    if artifact["bundle_sha256"] != actual_bundle:  # 按当前条件选择后续控制路径。
        raise ValueError("internal bundle digest mismatch")  # 遇到非法合同立即显式失败。
    if sha256(canonical_json35(manifest)).hexdigest() != artifact["manifest_sha256"]:  # 按当前条件选择后续控制路径。
        raise ValueError("manifest digest mismatch")  # 遇到非法合同立即显式失败。
    if canonical_state_digest35(state) != manifest["state_digest_sha256"]:  # 按当前条件选择后续控制路径。
        raise ValueError("canonical state digest mismatch")  # 遇到非法合同立即显式失败。
    validate_detr_contract35(manifest, state)  # 执行当前语句以推进本节示例。
    loaded = TinyDETR(**manifest["config"])  # 计算并保存当前步骤的中间状态。
    loaded.load_state_dict(state, strict=True)  # 计算并保存当前步骤的中间状态。
    return loaded.eval()  # 返回当前分支计算出的结果。

def resign_inside35(artifact):  # 定义本节可复用的核心函数。
    artifact["manifest"]["state_digest_sha256"] = canonical_state_digest35(artifact["state_dict"])  # 计算并保存当前步骤的中间状态。
    artifact["manifest_sha256"] = sha256(canonical_json35(artifact["manifest"])).hexdigest()  # 计算并保存当前步骤的中间状态。
    artifact["bundle_sha256"] = canonical_bundle_digest35(artifact["manifest"], artifact["state_dict"])  # 计算并保存当前步骤的中间状态。
    return artifact  # 返回当前分支计算出的结果。

artifact35 = make_detr_artifact(model35)  # 计算并保存当前步骤的中间状态。
PUBLISHER_REGISTRY35 = MappingProxyType({  # 计算并保存当前步骤的中间状态。
    (ARTIFACT_ID35, ARTIFACT_VERSION35): artifact35["bundle_sha256"]  # 执行当前语句以推进本节示例。
})  # 执行当前语句以推进本节示例。
loaded35 = load_trusted_detr(artifact35)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    original = model35(test_images[:2])["pred_logits"]  # 计算并保存当前步骤的中间状态。
    restored = loaded35(test_images[:2])["pred_logits"]  # 计算并保存当前步骤的中间状态。
assert torch.equal(original, restored)  # 用受控断言验证关键不变量。

# 攻击者整体换成 4-query 模型，并同时篡改 box 语义、重签所有 package 内摘要。
# 内部摘要完全自洽，但发布者登记的 bundle digest 没变，因此仍必须 fail closed。
forged_model35 = TinyDETR(num_queries=4)  # 计算并保存当前步骤的中间状态。
forged35 = deepcopy(artifact35)  # 计算并保存当前步骤的中间状态。
forged35["state_dict"] = clone_state35(forged_model35.state_dict())  # 计算并保存当前步骤的中间状态。
forged35["manifest"]["config"]["num_queries"] = 4  # 计算并保存当前步骤的中间状态。
forged35["manifest"]["model_state_schema"] = state_schema35(forged35["state_dict"])  # 计算并保存当前步骤的中间状态。
forged35["manifest"]["box_format"] = "absolute_xyxy"  # 计算并保存当前步骤的中间状态。
resign_inside35(forged35)  # 执行当前语句以推进本节示例。
assert forged35["bundle_sha256"] == canonical_bundle_digest35(forged35["manifest"], forged35["state_dict"])  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    load_trusted_detr(forged35)  # 执行当前语句以推进本节示例。
    raise AssertionError("self-signed whole replacement must fail")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。

try:  # 尝试执行可能失败的受控操作。
    PUBLISHER_REGISTRY35[(ARTIFACT_ID35, ARTIFACT_VERSION35)] = forged35["bundle_sha256"]  # 计算并保存当前步骤的中间状态。
    raise AssertionError("publisher registry must be immutable")  # 遇到非法合同立即显式失败。
except TypeError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。

## 13. 失败模式、复杂度与生产差距

- **把 target 按下标对齐 query**：预测顺序改变时 loss 改变；排列不变测试应失败。
- **空图跳过 classification loss**：模型学不会 no-object，线上会产生大量假阳性。
- **只在 attention mask padding**：padding 像素可能已通过 CNN 污染有效 feature；输入先清零，并按每层卷积感受野把 mask 保守传播到 2×2 feature。
- **全 mask 后 softmax**：会产生 NaN；应在进入 attention 前拒绝没有有效像素的样本。
- **box 单位混用**：normalized cxcywh、absolute xyxy 必须由 manifest 绑定。
- **穷举匹配上生产**：复杂度随目标数阶乘增长；实际使用 Hungarian/Jonker–Volgenant 等经过测试的多项式算法。

attention 的时间/显存复杂度约为 encoder $O(S^2D)$、decoder cross-attention $O(QSD)$；高分辨率 feature 会主导成本。生产系统还需多尺度特征、小目标策略、数据增强、分布式训练、mixed precision、COCO 风格 mAP、拥挤/遮挡/空图/OOD 分桶、阈值校准、延迟与显存压测。

### 论文来源

- Carion et al., [*End-to-End Object Detection with Transformers*](https://arxiv.org/abs/2005.12872), ECCV 2020.
- Vaswani et al., [*Attention Is All You Need*](https://arxiv.org/abs/1706.03762), NeurIPS 2017（scaled dot-product attention）。
- Kuhn, [*The Hungarian Method for the Assignment Problem*](https://doi.org/10.1002/nav.3800020109), 1955（最优二分匹配背景）。

这里复现 DETR 的集合预测骨架与工程合同，不代表复现 COCO 训练配方或论文指标。